# 13 — Local transformer entity extraction

Explore **GLiNER small v2.1 zero-shot extraction** on the same four entity types as notebook 11: KNOWLEDGE, SKILL, TITLE and DEGREE. This is a separate local model experiment, not a validated improvement over rules. The approved [transformer plan](../plan/2026-10-06-transformer-entity-extraction/README.md) authorizes a five-CV/five-JD smoke test followed by a bounded 100-CV/100-JD CPU run.

The default run samples only the frozen A2 training region at seed 0, uses a declared threshold of 0.5 and caps exploratory inference at 30 minutes. Compatible local results reload without repeating inference. A partial run must be read against its reported completion and coverage, not as a completed 200-document evaluation.

All inference is local. Real neural spans stay in ignored parquet because a predicted type cannot guarantee that a span contains no personal name. Default saved entity previews use authored synthetic text only.

In [1]:
import json
from pathlib import Path
import pandas as pd
from candidate_screener.config import PROCESSED

# Keep every saved output below the notebook's 2,000-character cap.
def show_summary(value):
    rendered = json.dumps(value, indent=2, default=str)
    print(rendered if len(rendered) <= 1900 else rendered[:1850] + "\n[output truncated]")

MODEL_DIR = PROCESSED / "extraction" / "models" / "gliner-small-v2.1"
SAMPLE_SIZE = 100  # per type; configurable after measuring this exploratory run
SEED = 0
THRESHOLD = 0.5
MAX_SECONDS = 1800


## Run or reload the real local experiment

The package verifies checkpoint artifacts and input provenance, preserves field-relative offsets through overlapping tokenizer windows, merges duplicate window predictions and writes model results separately from rules. Missing dependencies or model artifacts fail explicitly; they never silently substitute rule outputs.

The run manifest records checkpoint revision, hashes, labels, threshold, chunk policy, heuristic context version, resolved dependencies and device. Scores are finite **raw, uncalibrated model scores**; neither a score nor an extracted mention proves proficiency or job fit.

In [2]:
from candidate_screener.extraction.transformer_run import run_transformer, validate_transformer_run

result = run_transformer(sample_size=SAMPLE_SIZE, seed=SEED, threshold=THRESHOLD,
                         max_seconds=MAX_SECONDS, model_dir=MODEL_DIR)
show_summary({"run_id": result["run_id"], "local_evidence_directory": result["run_dir"]})


{
  "run_id": "2c591cc8bff7fc9e7189-20261005T164947399919Z",
  "local_evidence_directory": "/Users/user/Downloads/MTECH2526/PLP_project/intelligent-candidate-screener/data/processed/extraction/transformer-runs/2c591cc8bff7fc9e7189-20261005T164947399919Z"
}


In [3]:
show_summary({key: result["summary"][key] for key in ("status", "runtime_seconds", "requested_documents", "completed_documents", "gold_accuracy")})

{
  "status": "complete",
  "runtime_seconds": 183.81117159899986,
  "requested_documents": 200,
  "completed_documents": 200,
  "gold_accuracy": {
    "reason": "No validated local NER gold adapters",
    "status": "blocked"
  }
}


In [4]:
show_summary(result["summary"]["coverage"]["cv"] )

{
  "by_label": {
    "DEGREE": 19,
    "KNOWLEDGE": 813,
    "SKILL": 40,
    "TITLE": 205
  },
  "completed_documents": 100,
  "documents_with_entities": 100,
  "documents_with_label": {
    "DEGREE": 12,
    "KNOWLEDGE": 79,
    "SKILL": 21,
    "TITLE": 94
  },
  "entities": 1077,
  "fields_attempted": 400,
  "fields_complete": 400,
  "fields_incomplete": 0,
  "fields_requested": 400,
  "quarantined_predictions": 0,
  "raw_uncalibrated_scores": {
    "max": 0.9858959317207336,
    "mean": 0.7269153478196515,
    "min": 0.500027596950531
  },
  "requested_documents": 100
}


In [5]:
show_summary(result["summary"]["coverage"]["jd"] )

{
  "by_label": {
    "DEGREE": 23,
    "KNOWLEDGE": 517,
    "SKILL": 69,
    "TITLE": 146
  },
  "completed_documents": 100,
  "documents_with_entities": 99,
  "documents_with_label": {
    "DEGREE": 17,
    "KNOWLEDGE": 82,
    "SKILL": 42,
    "TITLE": 87
  },
  "entities": 755,
  "fields_attempted": 200,
  "fields_complete": 200,
  "fields_incomplete": 0,
  "fields_requested": 200,
  "quarantined_predictions": 0,
  "raw_uncalibrated_scores": {
    "max": 0.9687429666519165,
    "mean": 0.6833218640049562,
    "min": 0.5003635287284851
  },
  "requested_documents": 100
}


In [6]:
show_summary(result["summary"]["rule_comparison"])

{
  "completed_documents_compared": 200,
  "cv": {
    "exact_shared": 243,
    "rule_entities": 404,
    "rule_only": 161,
    "transformer_entities": 1077,
    "transformer_only": 834
  },
  "interpretation": "Exact span-label disagreement, not accuracy",
  "jd": {
    "exact_shared": 191,
    "rule_entities": 327,
    "rule_only": 136,
    "transformer_entities": 755,
    "transformer_only": 564
  }
}


## Rule comparison and operational validation

The rule baseline is computed on the **identical processed document subset**. Counts and disagreement show differences in extraction behaviour; agreement does not create gold labels or establish superiority. Field coverage and quarantine must be considered alongside mention coverage, especially for incomplete runs.

In [7]:
integrity = validate_transformer_run(result)
show_summary(integrity)


{
  "entities_checked": 1832,
  "unique_entity_ids": true,
  "exact_field_offsets": true,
  "training_only": true,
  "artifact_hashes": true,
  "checkpoint_hashes": true,
  "field_coverage_valid": true,
  "status": "complete",
  "score_replay": "Not required; raw scores can vary across devices"
}


## Synthetic model evidence

This example is authored text. The package returns exact spans into its immutable original field. Display redaction is separate from evidence validation and `assert_clean` runs inside the notebook. This illustrates model/context behaviour; it is not an accuracy test. The context layer remains a versioned rule heuristic, so this experiment is a **hybrid neural mention extractor with heuristic context**.

In [8]:
from candidate_screener.extraction.transformer import load_model, extract_transformer_document
from candidate_screener.extraction.display import preview_entities
from candidate_screener.annotation.redact import assert_clean, redact_text

synthetic_text = "No Python experience. Learning Kubernetes. Software engineer with a Master of Science."
synthetic_document = {"doc_id": "synthetic-transformer-demo", "source": "synthetic", "doc_type": "cv", "fields": {"Moreinfo": synthetic_text}}
model = load_model(MODEL_DIR, device="cpu")
synthetic_entities, synthetic_coverage = extract_transformer_document(
    synthetic_document, model, threshold=THRESHOLD, overlap_tokens=32)
assert all(synthetic_text[e["start"]:e["end"]] == e["surface"] for e in synthetic_entities)
synthetic_preview = preview_entities(synthetic_entities, limit=5)
assert_clean(pd.Series([redact_text(row["surface"]) for row in synthetic_preview], dtype="string"), "synthetic transformer preview")
show_summary(synthetic_preview)

if callable(getattr(model, "close", None)):
    model.close()


[
  {
    "surface": "Python",
    "label": "KNOWLEDGE",
    "field": "Moreinfo",
    "context_status": "negated",
    "doc_type": "cv"
  },
  {
    "surface": "Kubernetes",
    "label": "KNOWLEDGE",
    "field": "Moreinfo",
    "context_status": "learning",
    "doc_type": "cv"
  },
  {
    "surface": "Software engineer",
    "label": "TITLE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  },
  {
    "surface": "Master of Science",
    "label": "DEGREE",
    "field": "Moreinfo",
    "context_status": "unqualified",
    "doc_type": "cv"
  }
]


In [9]:
show_summary(synthetic_coverage)

{
  "Moreinfo": {
    "intervals": [
      [
        0,
        86
      ]
    ],
    "characters": 86,
    "chunks": 1,
    "planned_chunks": 1,
    "complete": true,
    "quarantined_intervals": [],
    "quarantined_predictions": 0,
    "interrupted": false
  }
}


## What this experiment can establish

This run can establish that a local transformer produces traceable predictions, its observed runtime and descriptive document/label coverage, and how its predictions differ from the limited seed rules. It does not establish precision, recall, F1, canonical mapping accuracy or a retrieval improvement.

DataTurks/SkillSpan restoration and validated adapters remain prerequisites for held-out extraction scoring. A later BERT/DistilBERT/DeBERTa token-classification experiment needs independently labelled spans, subword alignment, preserved splits/grouped folds and a training budget; rule predictions are not human gold. B1 automatically segmented skill blobs remain silver labels. No supervised training or new annotation runs here.

Inspect real records deliberately in the local `entities.parquet` at the evidence directory printed above. The saved-output helper suppresses unreviewed real neural spans rather than claiming regex redaction removes names. Any downstream retrieval comparison must report Recall@10 separately for `generic` and `targeted`; Precision@k must carry its ceiling and A13 uncertainty.